# 18 — Detector 1 Training

Train **Detector 1** for the Adaptive_Road_Vision object-detection stage.

This notebook uses the final BDD100K-derived training dataset and YOLO-format labels created in Notebook 10. Detector 1 is a clean baseline detector trained on the original/final training images without condition-specific enhancement.

The notebook:
- verifies image/label pairing,
- creates a YOLO-compatible dataset configuration,
- trains a YOLO detector,
- validates the best checkpoint,
- records training/evaluation artifacts,
- keeps validation/test data separate from training.

**Important:** This notebook expects the Ultralytics package. Install it from `requirements.txt` or run the installation cell if needed.

In [ ]:
from pathlib import Path
import os
import json
import random
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current, *list(current.parents)]
    for candidate in candidates:
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate
    fallback = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
    if fallback.exists():
        return fallback
    return current

PROJECT_ROOT = find_project_root()
def resolve_dataset_root():
    for cand in [
        PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
        PROJECT_ROOT / "datasets" / "final_condition_balanced",
        PROJECT_ROOT / "datasets" / "final",
    ]:
        if (cand / "metadata" / "final_dataset_metadata.csv").exists():
            return cand
        if (cand / "labels" / "train").exists() and any((cand / "labels" / "train").glob("*.txt")):
            return cand
    return PROJECT_ROOT / "datasets" / "final_condition_balanced_v2"

FINAL_DIR = resolve_dataset_root()
DATASET_ROOT = FINAL_DIR
RESULTS_DIR = PROJECT_ROOT / "results" / "detection"
MODEL_DIR = PROJECT_ROOT / "models" / "detection"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "detector_1"

for p in [
    RESULTS_DIR,
    MODEL_DIR,
    EXPERIMENT_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
IMAGE_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 50
PATIENCE = 10
WORKERS = 0
DEVICE = 0 if __import__("torch").cuda.is_available() else "cpu"

MODEL_NAME = "yolo11n.pt"
RUN_NAME = "detector_1_baseline"

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Model:", MODEL_NAME)


from PIL import Image
from IPython.display import display

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

## 1. Load class configuration

BDD100K detection labels use the project label set created during preprocessing. If a project-specific class configuration exists, it is loaded; otherwise the notebook uses the common BDD100K object classes.

The class IDs must match the YOLO label files generated by the earlier dataset-preparation notebooks.

In [ ]:
CLASS_CANDIDATES = [
    PROJECT_ROOT / 'configs' / 'detection.yaml',
    PROJECT_ROOT / 'configs' / 'dataset.yaml',
    DATASET_ROOT / 'dataset.yaml' if 'DATASET_ROOT' in globals() else FINAL_DIR / 'dataset.yaml',
]

class_names = None

for candidate in CLASS_CANDIDATES:
    if candidate.exists():
        try:
            import yaml

            with open(candidate, "r", encoding="utf-8") as f:
                cfg = yaml.safe_load(f)

            names = cfg.get("names") if isinstance(cfg, dict) else None

            if isinstance(names, dict):
                class_names = [
                    names[k]
                    for k in sorted(
                        names,
                        key=lambda x: int(x)
                        if str(x).isdigit()
                        else str(x)
                    )
                ]
            elif isinstance(names, list):
                class_names = names

            if class_names:
                print("Loaded classes from:", candidate)
                break

        except Exception as e:
            print("Could not parse", candidate, ":", e)

if not class_names:
    class_names = [
        "pedestrian",
        "rider",
        "car",
        "truck",
        "bus",
        "train",
        "motorcycle",
        "bicycle",
    ]

print("Classes:", class_names)
print("Number of classes:", len(class_names))


## 2. Verify final YOLO dataset

Expected structure:

```text
datasets/final/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

The notebook checks that labels exist, coordinates are valid, and image/label counts are sensible before training.

In [ ]:
IMAGE_DIR = FINAL_DIR / "images"
LABEL_DIR = FINAL_DIR / "labels"

assert IMAGE_DIR.exists(), f"Missing: {IMAGE_DIR}"
assert LABEL_DIR.exists(), f"Missing: {LABEL_DIR}"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def collect_images(split):
    folder = IMAGE_DIR / split
    return sorted([
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])

def label_for_image(image_path, split):
    return LABEL_DIR / split / f"{image_path.stem}.txt"

dataset_counts = {}

for split in ["train", "val", "test"]:
    images = collect_images(split)
    labels = [
        label_for_image(p, split)
        for p in images
    ]

    missing = [
        p for p in labels
        if not p.exists()
    ]

    dataset_counts[split] = {
        "images": len(images),
        "labels": sum(p.exists() for p in labels),
        "missing_labels": len(missing),
    }

    print(
        f"{split:>5}: "
        f"images={len(images):>6} | "
        f"labels={sum(p.exists() for p in labels):>6} | "
        f"missing={len(missing):>4}"
    )

print("\nDataset counts:")
display(pd.DataFrame(dataset_counts).T)

# Training and validation must contain images. Test is required for final held-out evaluation.
for required_split in ("train", "val", "test"):
    if dataset_counts[required_split]["images"] == 0:
        raise RuntimeError(
            f"No images found for split '{required_split}' under {IMAGE_DIR / required_split}. "
            "Check the final dataset directory and split folder names."
        )

# Missing label files can silently turn annotated images into background samples.
missing_label_total = sum(
    dataset_counts[split]["missing_labels"] for split in ("train", "val", "test")
)
if missing_label_total:
    raise RuntimeError(
        f"Found {missing_label_total} images without matching YOLO label files. "
        "Fix image/label pairing before training so objects are not silently treated as background."
    )

In [ ]:
def validate_yolo_label(label_path, n_classes):
    errors = []

    try:
        text = label_path.read_text(
            encoding="utf-8"
        ).strip()
    except Exception as e:
        return [f"read_error:{e}"]

    if not text:
        return errors

    for line_no, line in enumerate(
        text.splitlines(),
        start=1
    ):
        parts = line.split()

        if len(parts) != 5:
            errors.append(
                f"line_{line_no}:expected_5_values"
            )
            continue

        try:
            class_id = int(float(parts[0]))
            values = [
                float(v)
                for v in parts[1:]
            ]
        except ValueError:
            errors.append(
                f"line_{line_no}:non_numeric"
            )
            continue

        if not (
            0 <= class_id < n_classes
        ):
            errors.append(
                f"line_{line_no}:bad_class_{class_id}"
            )

        if not all(
            0.0 <= v <= 1.0
            for v in values
        ):
            errors.append(
                f"line_{line_no}:coordinate_out_of_range"
            )

        if values[2] <= 0 or values[3] <= 0:
            errors.append(
                f"line_{line_no}:non_positive_box"
            )

    return errors


label_errors = []

for split in ["train", "val", "test"]:
    for image_path in collect_images(split):
        label_path = label_for_image(
            image_path,
            split
        )

        if not label_path.exists():
            continue

        errors = validate_yolo_label(
            label_path,
            len(class_names)
        )

        if errors:
            label_errors.append({
                "split": split,
                "label_path": str(label_path),
                "errors": ";".join(errors),
            })

errors_df = pd.DataFrame(label_errors)

print(
    "Label files with validation errors:",
    len(errors_df)
)

if len(errors_df):
    display(errors_df.head(20))
else:
    print("No YOLO label-format errors detected.")

if len(errors_df):
    raise RuntimeError(
        f"Found {len(errors_df)} invalid YOLO label files. Correct these before training."
    )

## 3. Create YOLO dataset YAML

Ultralytics needs a dataset YAML pointing to the project image folders and class names.

The generated YAML is saved under:

`experiments/detector_1/detector_1_dataset.yaml`

In [ ]:
import yaml

dataset_yaml_path = (
    EXPERIMENT_DIR /
    "detector_1_dataset.yaml"
)

dataset_yaml = {
    "path": str(FINAL_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        i: name
        for i, name in enumerate(class_names)
    },
}

with open(
    dataset_yaml_path,
    "w",
    encoding="utf-8"
) as f:
    yaml.safe_dump(
        dataset_yaml,
        f,
        sort_keys=False
    )

print(dataset_yaml_path)
print(
    dataset_yaml_path.read_text(
        encoding="utf-8"
    )
)

## 4. Load Ultralytics

If Ultralytics is already installed, this cell uses it directly. Otherwise uncomment the installation command.

For a clean environment, installing from your project's `requirements.txt` is preferred.

In [ ]:
try:
    from ultralytics import YOLO
    import ultralytics

    print(
        "Ultralytics:",
        ultralytics.__version__
    )

except ImportError:
    print(
        "Ultralytics is not installed."
    )
    print(
        "Run: pip install ultralytics"
    )
    raise

## 5. Train Detector 1

Detector 1 is the baseline YOLO detector.

The validation split is used by Ultralytics for model selection and metrics. The test split is not used for gradient updates or checkpoint selection.

In [ ]:
model = YOLO(MODEL_NAME)

train_args = {
    "data": str(dataset_yaml_path),
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "workers": WORKERS,
    "device": DEVICE,
    "project": str(EXPERIMENT_DIR),
    "name": RUN_NAME,
    "exist_ok": True,
    "pretrained": True,
    "optimizer": "auto",
    "patience": PATIENCE,
    "seed": SEED,
    "verbose": True,
    "plots": True,
}

print(json.dumps(train_args, indent=2, default=str))

In [ ]:
train_results = model.train(
    **train_args
)

print("Training complete.")
print("Run directory:", EXPERIMENT_DIR / RUN_NAME)

## 6. Locate the best checkpoint

Ultralytics normally saves:

```text
experiments/detector_1/detector_1/weights/
├── best.pt
└── last.pt
```

The notebook copies these to the project's model directory for consistent downstream access.

In [ ]:
RUN_DIR = EXPERIMENT_DIR / RUN_NAME
WEIGHTS_DIR = RUN_DIR / "weights"

best_source = WEIGHTS_DIR / "best.pt"
last_source = WEIGHTS_DIR / "last.pt"

assert best_source.exists(), (
    f"Best checkpoint not found: {best_source}"
)

best_model_path = (
    MODEL_DIR /
    "detector_1_best.pt"
)

last_model_path = (
    MODEL_DIR /
    "detector_1_last.pt"
)

shutil.copy2(
    best_source,
    best_model_path
)

if last_source.exists():
    shutil.copy2(
        last_source,
        last_model_path
    )

print("Best:", best_model_path)
print("Last:", last_model_path)

## 7. Validation evaluation

Run validation using the saved best checkpoint and collect the standard detection metrics reported by Ultralytics.

In [ ]:
best_model = YOLO(
    str(best_model_path)
)

val_metrics = best_model.val(
    data=str(dataset_yaml_path),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    project=str(EXPERIMENT_DIR),
    name="detector_1_validation",
    exist_ok=True,
    plots=True,
    verbose=True,
)

print(val_metrics)

## 8. Test evaluation

The test set is evaluated only after training and model selection are complete. This provides the held-out baseline detector result for later comparison with Detector 2–4.

In [ ]:
test_metrics = best_model.val(
    data=str(dataset_yaml_path),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    project=str(EXPERIMENT_DIR),
    name="detector_1_test",
    exist_ok=True,
    plots=True,
    verbose=True,
)

print(test_metrics)

## 9. Extract summary metrics

Save a compact JSON report that later detector-comparison notebooks can consume.

In [ ]:
def extract_detection_metrics(metrics):
    output = {}

    # Ultralytics exposes these through .results_dict
    # for common detection tasks.
    if hasattr(metrics, "results_dict"):
        for key, value in metrics.results_dict.items():
            try:
                output[key] = float(value)
            except Exception:
                output[key] = str(value)

    # Include speed information when available.
    if hasattr(metrics, "speed"):
        try:
            output["speed"] = {
                k: float(v)
                for k, v in metrics.speed.items()
            }
        except Exception:
            pass

    return output


val_summary = extract_detection_metrics(
    val_metrics
)

test_summary = extract_detection_metrics(
    test_metrics
)

summary = {
    "detector": "detector_1",
    "model": MODEL_NAME,
    "dataset_yaml": str(dataset_yaml_path),
    "best_checkpoint": str(best_model_path),
    "last_checkpoint": str(last_model_path),
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_requested": EPOCHS,
    "classes": class_names,
    "dataset_counts": dataset_counts,
    "validation_metrics": val_summary,
    "test_metrics": test_summary,
}

summary_path = (
    RESULTS_DIR /
    "detector_1_training_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(
    json.dumps(
        summary,
        indent=2
    )
)

print("Saved:", summary_path)

## 10. Save class-wise test metrics when available

Ultralytics exposes per-class metrics through the detection metrics object. This cell attempts to record them without assuming a particular Ultralytics version.

In [ ]:
class_rows = []

try:
    box = test_metrics.box

    precision = (
        np.asarray(box.p)
        if hasattr(box, "p")
        else None
    )

    recall = (
        np.asarray(box.r)
        if hasattr(box, "r")
        else None
    )

    f1 = (
        np.asarray(box.f1)
        if hasattr(box, "f1")
        else None
    )

    ap50 = (
        np.asarray(box.ap50)
        if hasattr(box, "ap50")
        else None
    )

    ap = (
        np.asarray(box.ap)
        if hasattr(box, "ap")
        else None
    )

    for i, name in enumerate(class_names):
        row = {
            "class_id": i,
            "class_name": name,
        }

        if precision is not None and i < len(precision):
            row["precision"] = float(precision[i])

        if recall is not None and i < len(recall):
            row["recall"] = float(recall[i])

        if f1 is not None and i < len(f1):
            row["f1"] = float(f1[i])

        if ap50 is not None and i < len(ap50):
            row["ap50"] = float(ap50[i])

        if ap is not None and i < len(ap):
            # AP can be an array of IoU values.
            row["ap50_95"] = float(
                np.mean(ap[i])
                if np.ndim(ap[i]) > 0
                else ap[i]
            )

        class_rows.append(row)

except Exception as e:
    print(
        "Could not extract class-wise metrics:",
        e
    )

class_metrics_df = pd.DataFrame(
    class_rows
)

class_metrics_path = (
    RESULTS_DIR /
    "detector_1_test_class_metrics.csv"
)

class_metrics_df.to_csv(
    class_metrics_path,
    index=False
)

if len(class_metrics_df):
    display(
        class_metrics_df.round(4)
    )

print(
    "Saved:",
    class_metrics_path
)

## 11. Training curves and artifact inspection

Display the standard Ultralytics training result plot when available and list the important artifacts.

In [ ]:
results_plot = RUN_DIR / "results.png"

if results_plot.exists():
    display(
        Image.open(results_plot)
    )
else:
    print(
        "Training results plot not found:",
        results_plot
    )

artifact_rows = []

for path in [
    best_model_path,
    last_model_path,
    summary_path,
    class_metrics_path,
    dataset_yaml_path,
]:
    artifact_rows.append({
        "artifact": str(path),
        "exists": path.exists(),
        "size_mb": (
            path.stat().st_size / (1024 ** 2)
            if path.exists()
            else None
        ),
    })

display(
    pd.DataFrame(artifact_rows)
)

## 12. Final detector-1 report

Detector 1 is the baseline detector for the next notebooks. Detector 2–4 can use the same dataset YAML and evaluation protocol so their results remain comparable.

The key model artifact is:

`models/detection/detector_1_best.pt`

In [ ]:
final_report = {
    "detector_id": "detector_1",
    "purpose": "baseline object detector",
    "checkpoint": str(best_model_path),
    "validation_report": val_summary,
    "test_report": test_summary,
    "class_metrics_csv": str(class_metrics_path),
    "training_run": str(RUN_DIR),
    "dataset_yaml": str(dataset_yaml_path),
}

final_report_path = (
    RESULTS_DIR /
    "detector_1_final_report.json"
)

with open(
    final_report_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_report,
        f,
        indent=2
    )

print(
    json.dumps(
        final_report,
        indent=2
    )
)

print(
    "\nDetector 1 checkpoint:",
    best_model_path
)